# Robust AI-Image Detector — Free Kaggle Training

This notebook clones the public repository, prepares a balanced SID_Set subset, trains the clean and robust models, evaluates the final checkpoint, and packages the outputs for download. Enable a Kaggle GPU and internet access before running all cells.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/LINGSIHAN/TikTok-Hackathon-Track-5.git"
BRANCH = "master"
PROJECT_DIR = Path("/kaggle/working/TikTok-Hackathon-Track-5")
SUBSET_SIZE = 10_000
SEED = 42

def run(*args):
    print("+", " ".join(map(str, args)))
    subprocess.run([str(arg) for arg in args], check=True)

In [ ]:
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR)
os.chdir(PROJECT_DIR)
run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Kaggle choose Settings > Accelerator > GPU, then restart.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
run(
    sys.executable,
    "scripts/prepare_sid_subset.py",
    "--total", str(SUBSET_SIZE),
    "--seed", str(SEED),
)

In [ ]:
run(sys.executable, "-m", "src.training.train", "--config", "configs/train_clean.yaml")

In [ ]:
run(sys.executable, "-m", "src.training.train", "--config", "configs/train_robust.yaml")

In [ ]:
run(
    sys.executable, "-m", "src.evaluation.evaluate",
    "--manifest", "data/processed/manifest.csv",
    "--checkpoint", "artifacts/checkpoints/model.safetensors",
    "--split", "test",
    "--output-dir", "artifacts/metrics",
    "--device", "cuda",
)

In [ ]:
export_dir = Path("/kaggle/working/export")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)

for source in [
    Path("artifacts/checkpoints/model.safetensors"),
    Path("artifacts/checkpoints/model_metadata.json"),
]:
    if source.exists():
        shutil.copy2(source, export_dir / source.name)

metrics_dir = Path("artifacts/metrics")
if metrics_dir.exists():
    shutil.copytree(metrics_dir, export_dir / "metrics", dirs_exist_ok=True)

archive = shutil.make_archive("/kaggle/working/hackathon_export", "zip", export_dir)
print("Download this file from Kaggle Output:", archive)